# Router study: is fine-tuning worth it?
The router fine-tuning study (brief section 2), as it was run (VS Code with an NVIDIA GPU; Colab also works). It copies the repository to read the data and sends nothing back; the results stay in your work folder. The committed results are in fine_tuning/results/router_study/ and the write-up in fine_tuning/README_router_study.md. It answers four questions, changing one thing at a time:

1. **Is the current router trained at all?** It checks the live router's weights against a fresh download of the pretrained model.
2. **Which training settings work best?** Every combination of learning rate and batch size, each with its best number of epochs, on validation only.
3. **Does fine-tuning change the model, and how does it improve during training?** Per-epoch loss, accuracy, macro-F1, per-route F1 and weight change; a **frozen** version (pretrained weights untouched, only the last layer trained) as the no-fine-tuning baseline; the sentence space before and after training.
4. **How does it compare with prompted models?** A small (gpt-4o-mini) and a large (gpt-4o) model, each with and without example questions, on the same questions. With 95% confidence intervals, significance tests, per-route F1, confusion matrices, speed and cost.
5. **When does it pay off?** Monthly cost at 1k / 100k / 1M questions, including the cost of misrouted questions, and the break-even volume.

The test sets are used **once**, at the end (step 11).

## Before you start (once)
This version runs **on your own computer in VS Code**, using your NVIDIA GPU. Nothing goes to Colab, and nothing is sent back to GitHub.

1. **Install** (skip anything you already have): **Python 3.12** from python.org (tick *"Add python.exe to PATH"*), **Git for Windows**, and in VS Code the **Python** and **Jupyter** extensions.
2. **Work folder:** step 1 uses a `cusma_router` folder in your home folder (outside OneDrive); change `WORK` there to use another.
3. **Key:** the LLM routers need an OpenRouter key. Create a file called `.env` in the work folder with one line, `OPENROUTER_API_KEY=` followed by your key (no quotes). The file stays on your computer and is never added to the repo. Without it, step 1 asks for the key (it is never printed).
4. **A clean Python environment:** in VS Code press **Ctrl+Shift+P → "Python: Create Environment" → Venv → Python 3.12**. Then open this notebook and pick that environment as the kernel (top right, **Select Kernel**).

Run the cells in order (Shift+Enter).

**Time and cost:** the first run of step 1 takes 5–10 minutes (it downloads PyTorch, about 2.5 GB). Steps 2–9 take about 45–60 minutes on an RTX 4000 Ada laptop GPU, and about $8 in API costs on the full validation sets, mostly the large model. For a quick trial first, set `LIMIT = 50` and a smaller `SEARCH` in step 3 (about 10 minutes, $0.40).

## 1. Setup: keys, copy the repo, install (5–10 minutes the first time)
Nothing to change here, except `WORK` if you want another folder.

In [ ]:
import os, subprocess, sys, json, getpass
from pathlib import Path

WORK = Path.home() / "cusma_router"     # your work folder (outside OneDrive); change if you like
WORK.mkdir(parents=True, exist_ok=True)

def secret(name):
    """From the environment, else WORK/.env, else ask. Never printed."""
    if os.environ.get(name):
        return os.environ[name]
    env = WORK / ".env"
    if env.exists():
        for line in env.read_text(encoding="utf-8").splitlines():
            if line.strip().startswith(name + "="):
                return line.split("=", 1)[1].strip().strip('"').strip("'")
    return getpass.getpass(f"{name}: ")

os.environ["OPENROUTER_API_KEY"] = secret("OPENROUTER_API_KEY")

REPO_URL  = "https://github.com/JP-DSmith/woodb-agent-project.git"
REPO_ROOT = WORK / "woodb-agent-project"
BRANCH    = "backup/router-finetune"
import base64
AUTH = []   # the repository is public: no token needed to read it
if not (REPO_ROOT / ".git").exists():
    subprocess.run(["git", *AUTH, "clone", "--quiet", "--branch", "dev", REPO_URL, str(REPO_ROOT)], check=True)
else:   # already copied: bring it up to date with dev (discards local changes in the copy)
    subprocess.run(["git", *AUTH, "fetch", "--quiet", "origin", "dev"], cwd=REPO_ROOT, check=True)
    subprocess.run(["git", "checkout", "--quiet", "--force", "FETCH_HEAD"], cwd=REPO_ROOT, check=True)
print("Repo at", REPO_ROOT, subprocess.run(["git", "log", "-1", "--format=%h %s"], cwd=REPO_ROOT,
                                           capture_output=True, text=True).stdout.strip())

pip = [sys.executable, "-m", "pip", "install", "-q"]
has_cuda = subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.is_available())"],
                          capture_output=True, text=True).stdout.strip() == "True"
if not has_cuda:   # the standard Windows PyTorch is CPU-only: install the NVIDIA (CUDA) build
    print("Installing PyTorch with NVIDIA support (about 2.5 GB, a few minutes)...")
    subprocess.run(pip + ["--upgrade", "--force-reinstall", "torch", "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
subprocess.run(pip + ["-r", "requirements.txt", "transformers", "datasets", "accelerate", "safetensors", "onnx",
                      "scikit-learn", "matplotlib", "pandas", "pytest"], cwd=REPO_ROOT, check=True)
os.chdir(REPO_ROOT); sys.path.insert(0, str(REPO_ROOT))
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      "NONE - check the NVIDIA driver, then restart the kernel and run this cell again")
print("Ready. Nothing will be sent back to GitHub.")

## 2. Check the study's code and run its tests

In [ ]:
# The study's code is in the repository (fine_tuning/backup/). Run its tests.
missing = [f for f in ["fine_tuning/backup/train_finetuned.py", "fine_tuning/backup/routers.py", "fine_tuning/backup/economics.py",
                       "fine_tuning/backup/analysis.py", "fine_tuning/backup/experiments.py"] if not (REPO_ROOT / f).exists()]
if missing:
    raise RuntimeError(f"Missing from this copy of the repository: {missing}")
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "testing/test_backup_router.py"], capture_output=True, text=True, cwd=REPO_ROOT)
print(tests.stdout[-600:], tests.stderr[-600:])
if tests.returncode:
    raise RuntimeError("Tests failed. Send the output above.")

## 3. Settings
`BASE` is the starting point. Step 5 searches `SEARCH` (learning rate × batch size) and replaces those values with the best on validation. Every version then trains for up to `SEARCH_EPOCHS` epochs and keeps its own best epoch (versions with less data need more). `VERSIONS` lists what each trained version changes, one thing at a time. Every run adds rows to `fine_tuning/results/backup/tuning_log.csv`.

In [ ]:
import importlib
from fine_tuning.backup import train_finetuned as tf, routers as br, economics as econ, analysis as an, experiments as ex
for m in (tf, br, econ, an, ex):
    importlib.reload(m)

BASE = {**tf.DEFAULTS,
        "base_model": "sentence-transformers/all-MiniLM-L6-v2",   # the same pretrained model the current router uses
        "epochs": 4, "learning_rate": 5e-5, "batch_size": 32, "max_length": 64, "seed": 894}
SEARCH = {"learning_rate": [2e-5, 5e-5, 1e-4], "batch_size": [16, 32]}   # every combination is tried
SEARCH_EPOCHS = 6                 # each search run keeps its best epoch, up to this many
VERSIONS = dict(ex.VERSIONS)      # fine-tuned / frozen encoder / original data / 25% / 50% data
EXTRA_BASE_MODELS = ["distilbert-base-uncased", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"]   # bigger base models, same settings: justify the base model choice
SMALL_MODEL, LARGE_MODEL = "openai/gpt-4o-mini", "openai/gpt-4o"
FEW_SHOT = 3                      # examples per route for the "+ examples" LLM routers (from training only)
LIMIT = None                      # questions per validation set; 50 for a quick trial
TRY_NOTE = "full run"
OUT = REPO_ROOT / "fine_tuning" / "results" / "backup"
MODELS = str(WORK / "router_models")      # trained models, outside the repo
print(json.dumps(VERSIONS, indent=1))

## 4. Is the current router trained? (about 1 minute)
Compares every weight in the live router's model file with a fresh download of the pretrained model.

In [ ]:
live_check = an.live_router_matches_pretrained(str(REPO_ROOT / "models/router/model.onnx"), BASE["base_model"])
print(f"{live_check['exact_identical']} of {live_check['exact_compared']} full-precision weight tables identical to the download "
      f"(largest difference {live_check['exact_max_difference']}); {live_check['compressed_compared']} compressed tables, "
      f"correlation {live_check['compressed_min_correlation']}")

## 5. Choose the training settings (about 15 minutes)
Trains every combination in `SEARCH` on the training set and scores it on validation after each epoch. The best combination, with its best epoch count, is used for every version in step 6.

In [ ]:
from IPython.display import Markdown, display, Image
search_results = ex.search(BASE, MODELS, SEARCH, epochs=SEARCH_EPOCHS)
BEST = ex.best_settings(BASE, search_results, max_epochs=SEARCH_EPOCHS)   # each version keeps its own best epoch
OUT.mkdir(parents=True, exist_ok=True)
an.chart_search(search_results, OUT / "hyperparameter_search.png")
display(Markdown(an.search_table(search_results)))
display(Image(str(OUT / "hyperparameter_search.png")))
print("Chosen:", {k: BEST[k] for k in ("learning_rate", "batch_size")}, f"- each version trains up to {SEARCH_EPOCHS} epochs and keeps its best")

## 6. Train the versions (about 10–15 minutes on a T4)
Each prints its progress per epoch. Epoch 0 is before any training.

In [ ]:
ALL_VERSIONS = {**VERSIONS, **{f"fine-tuned, {m.split('/')[-1]}": {"base_model": m} for m in EXTRA_BASE_MODELS}}
infos = ex.train_versions(BEST, MODELS, ALL_VERSIONS)
display(Markdown(an.training_table(infos)))
display(Markdown("**Fine-tuned, epoch by epoch**\n\n" + an.epoch_table(infos["fine-tuned"])))

## 7. Compare all routers on validation (about 10 minutes, mostly the large model)

In [ ]:
from evaluation import compare_routers as cr
_prices = cr._prices()   # fallback prices, used only if OpenRouter doesn't report a cost
cr._prices = lambda: {**_prices, "openai/gpt-4o": {"prompt_usd_per_token": 0.0000025, "completion_usd_per_token": 0.00001}}

ROUTERS = {"current method": cr.nearest_neighbour_router(),
           "frozen + classifier": br.make_probe_router(BASE["base_model"])}   # the fair no-fine-tuning baseline
ROUTERS |= {name: br.make_finetuned_router(f"{MODELS}/{ex.slug(name)}") for name in infos}
LLM_CACHE = WORK / "llm_cache.json"   # saved LLM answers: a rerun doesn't pay for the same question twice
ROUTERS |= {"llm small": br.make_llm_router(SMALL_MODEL, 0, cache_path=LLM_CACHE),
            "llm small + examples": br.make_llm_router(SMALL_MODEL, FEW_SHOT, cache_path=LLM_CACHE),
            "llm large": br.make_llm_router(LARGE_MODEL, 0, cache_path=LLM_CACHE),
            "llm large + examples": br.make_llm_router(LARGE_MODEL, FEW_SHOT, cache_path=LLM_CACHE)}
FOCUS = ("val_new_wording", "val_new_style")
sets = {k: v for k, v in cr.load_sets(cr.VALIDATION_SETS).items() if k in FOCUS}
if LIMIT:   # a random sample (the files are grouped by template, so the first N would cover only a few routes)
    import random
    sets = {k: random.Random(894).sample(v, min(LIMIT, len(v))) for k, v in sets.items()}
rows = []
for name, fn in ROUTERS.items():
    print("Routing with", name, "...")
    rows += cr.run_router(name, fn, sets)
    if hasattr(fn, "save"):
        fn.save()
    if hasattr(fn, "seconds"):     # reused answers: report the original call's time, not ~0 s
        br.restore_seconds(rows, name, fn)
    if getattr(fn, "unpriced", 0):
        print(f"  note: {fn.unpriced} reused answers were saved without their cost (older cache)")
display(Markdown(an.comparison_table(an.group_by_router(rows))))
print("frozen + classifier:", ROUTERS["frozen + classifier"].info)

## 8. The sentence space before and after fine-tuning (about 1 minute)

In [ ]:
from fine_tuning.backup.train_finetuned import load
emb = ex.embedding_check(BASE["base_model"], None, f"{MODELS}/{ex.slug('fine-tuned')}", load("val_new_style"),
                         OUT / "embeddings_before_after.png")
print(emb)
display(Image(str(OUT / "embeddings_before_after.png")))

## 9. Economics and the full report
Uses the measured accuracy and cost per question, plus the stated assumptions in `INPUTS` (change them here, and say which you used in the report).

In [ ]:
INPUTS = dict(econ.INPUTS)        # e.g. INPUTS["misroute_usd"] = 8.0
def summarise(rows, sets_used):
    out = []
    for name, rs in an.group_by_router(rows, set(sets_used)).items():
        m = an.metrics(rs)
        out.append({"name": name, "kind": "finetuned" if name.startswith(("fine-tuned", "frozen")) else
                    ("llm" if name.startswith("llm") else "current"), "accuracy": m["accuracy"],
                    "cost_usd": sum(r["cost_usd"] for r in rs) / len(rs), "seconds": sum(r["seconds"] for r in rs) / len(rs)})
    return out
economics_md = econ.table(summarise(rows, FOCUS), infos["fine-tuned"]["training_seconds"], INPUTS)
report = ex.write_report(OUT, infos, rows, FOCUS, emb, economics_md, live_check=live_check,
                         search_results=search_results)
display(Markdown(report.split("![")[0]))
for chart in ["hyperparameter_search", "weight_change", "learning_curves", "f1_per_route", "confusion_matrices", "cost_vs_quality", "training_size"]:
    if (OUT / f"{chart}.png").exists():
        display(Image(str(OUT / f"{chart}.png")))
display(Markdown(economics_md))

import csv, datetime
log = OUT / "tuning_log.csv"
new = not log.exists()
with log.open("a", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    if new:
        w.writerow(["time", "router", "note", "settings", "limit", "accuracy", "macro_f1"])
    for name, rs in an.group_by_router(rows).items():
        m = an.metrics(rs)
        settings = json.dumps({k: v for k, v in {**BEST, **ALL_VERSIONS[name]}.items() if k != "model_dir"}) if name in ALL_VERSIONS else \
            (json.dumps({"model": LARGE_MODEL if "large" in name else SMALL_MODEL,
                         "few_shot": FEW_SHOT if "examples" in name else 0}) if name.startswith("llm") else "")
        w.writerow([datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), name, TRY_NOTE, settings, LIMIT or "all",
                    f"{m['accuracy']:.4f}", f"{m['macro_f1']:.4f}"])

## 10. Collect the results
Copies the report, all the charts, the training details and the tuning log into one zip in your work folder (`results_<date>.zip`). Nothing goes to GitHub.

In [ ]:
import shutil, datetime
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
bundle = shutil.make_archive(str(WORK / f"results_{stamp}"), "zip", OUT.parent, ".")
print("Contents:", sorted(str(p.relative_to(OUT.parent)) for p in OUT.parent.rglob("*") if p.is_file() and "backup" in str(p)))
print("Saved:", bundle)

## 11. FINAL: the test sets, run ONCE when tuning is finished
Adds the holdout, team-written and hand-written questions, plus the live router itself. Report these numbers, and don't change anything afterwards (if you do, log it in T35). Then run step 10 again to collect them.

In [ ]:
RUN_FINAL = False    # True only when tuning is finished
if RUN_FINAL:
    final_sets = cr.load_sets(list(cr.FINAL_SETS))
    final_routers = {**ROUTERS, "live router": cr.load_router("live")[1]}
    final_rows = []
    for name, fn in final_routers.items():
        print("Routing with", name, "...")
        final_rows += cr.run_router(name, fn, final_sets)
        if hasattr(fn, "save"):
            fn.save()
        if hasattr(fn, "seconds"):     # reused answers: report the original call's time, not ~0 s
            br.restore_seconds(final_rows, name, fn)
        if getattr(fn, "unpriced", 0):
            print(f"  note: {fn.unpriced} reused answers were saved without their cost (older cache)")
    OUT_FINAL = OUT.parent / "backup_final"
    final_econ = econ.table(summarise(final_rows, final_sets), infos["fine-tuned"]["training_seconds"], INPUTS)
    final_report = ex.write_report(OUT_FINAL, infos, final_rows, list(final_sets), emb, final_econ, final=True,
                                   live_check=live_check, search_results=search_results)
    display(Markdown(final_report.split("![")[0]))
    for chart in ["f1_per_route", "confusion_matrices", "cost_vs_quality"]:
        display(Image(str(OUT_FINAL / f"{chart}.png")))
    display(Markdown(final_econ))